# Exploring AURORA object-store data

## Goal
Find the available data, inspect its structure, and read small examples of surface meteorology, CL61 backscatter, and power telemetry. This is a tutorial for researchers, not a completeness audit or a scientific QC product.

The source repository is [TONe ICO data access](https://github.com/Troll-Observing-Network/integrated-cloud-observatory-data-access). AURORA uses ordinary Zarr in the examples below; TONe's Icechunk storage assumptions do not carry over.

## Setup
Run `uv sync --locked`, configure an authorised AWS profile or environment credentials, then start `uv run jupyter lab` from this directory. See the README. Never paste credentials into a notebook.

All times are UTC. Data remain on the object store; only metadata and selected chunks are read. This notebook writes local evidence to the ignored `reports/` folder.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from aurora_access import Store, inventory, save_inventory, example, error_status, AccessError

started = time.monotonic()
store = Store()
report_dir = Path("reports")
report_dir.mkdir(exist_ok=True)
HOURS = 1  # May be increased to 24; each selected array is limited to 100 MiB.
results = []
plt.rcParams.update({"figure.figsize": (11, 4), "font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "savefig.dpi": 140})

## Steps
### 1. Discover data families
The inventory lists one hierarchy level at a time and stops at Zarr roots. Raw files and internal evidence receive shallow listings. Large collections of dated product stores are represented by their prefixes and three lexicographically last members. These names are not a measurement-time coverage estimate.

The inventory is private local output. An accessible metadata object does not prove all data chunks are present.

In [ ]:
discovery = inventory(store)
save_inventory(discovery, report_dir)
print("Observed at:", discovery["observed_at_utc"])
print("Unvisited queued prefixes:", discovery["unvisited_prefixes"])
rows = discovery["datasets"]
pd.DataFrame([{k: r.get(k, "") for k in ("area", "prefix", "status", "format")} for r in rows]).head(35)

### 2. Inspect array metadata
Shapes, dimensions, units and chunk sizes are copied from consolidated Zarr metadata. Missing units remain missing; no calibration or unit assumptions are introduced.

In [ ]:
array_rows = []
for row in rows:
    for name, details in row.get("arrays", {}).items():
        array_rows.append({"store": row["prefix"], "variable": name, **details})
metadata_table = pd.DataFrame(array_rows)
metadata_table.head(20)

### 3. Read and plot bounded examples
For each store, inspect at most 200,000 recent timestamps and select the first hour of the newest prior elapsed UTC day with observations. This does **not** establish complete daily coverage. If the time tail is insufficient or unordered, the example reports a blocker.

Select the variable and, for CL61, range 0–3,000 m before computation. Remote chunks can be larger than the selected array; 100 MiB limits the estimated decoded selection, not network transfer. Coordinates and metadata use separate bounded reads. Scatter marks retain missing intervals without joining across gaps. CL61 uses instrument range, not an assumed height above ground.

In [ ]:
def run_example(name):
    try:
        values, evidence = example(store, name, hours=HOURS)
    except Exception as exc:
        result = {"example": name, "status": error_status(exc)}
        if isinstance(exc, AccessError):
            result["detail"] = str(exc)
        results.append(result)
        print(result)
        return
    evidence["status"] = "readable"
    results.append(evidence)
    print(json.dumps(evidence, indent=2))
    fig, ax = plt.subplots(layout="constrained")
    times = mdates.date2num(pd.DatetimeIndex(values.time.values).to_pydatetime())
    units = evidence["units"]
    if "range" in values.dims:
        matrix = values.transpose("time", "range").values
        x = np.broadcast_to(times[:, None], matrix.shape)
        y = np.broadcast_to(values["range"].values[None, :], matrix.shape)
        finite = np.isfinite(matrix)
        points = ax.scatter(x[finite], y[finite], c=matrix[finite], s=2, cmap="viridis", rasterized=True)
        fig.colorbar(points, ax=ax, label=f"{evidence['variable']} ({units})")
        ax.set_ylabel(f"Instrument range ({values['range'].attrs.get('units', 'units absent')})")
    else:
        ax.scatter(times, values.values, s=5, color="#28658c", rasterized=True)
        ax.set_ylabel(f"{evidence['variable']} ({units})")
    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.set_xlabel("UTC on " + evidence["window_start_utc"][:10])
    ax.set_title(f"AURORA {name}: {HOURS:g}-hour sample")
    ax.grid(alpha=0.2)
    fig.savefig(report_dir / f"{name}.png")
    plt.show()
    plt.close(fig)

### Surface meteorology

In [ ]:
run_example("meteorology")

### CL61 attenuated backscatter

In [ ]:
run_example("ceilometer")

### Power telemetry

In [ ]:
run_example("power")

## Checks
Readable means the selected variable contains finite values in the stated window. It does not mean the source is current, complete, calibrated, or scientifically quality controlled. Inventory metadata may describe a different update instant from a later chunk read because these are mutable operational stores.

Review the per-example timestamps and finite-value counts before using the plots. Surface and power units may be absent from source metadata; the axes flag this explicitly. Unreadable examples remain listed, with no silent substitution.

In [ ]:
store.close()
validation = {"checked_at_utc": datetime.now(timezone.utc).isoformat(),
              "elapsed_seconds": round(time.monotonic() - started, 2),
              "examples": results}
(report_dir / "validation.json").write_text(json.dumps(validation, indent=2))
pd.DataFrame(results)[["example", "status"]]

## Next steps
Use `reports/inventory.json` to choose another dataset and inspect its metadata before reading observations. Increase `HOURS` only when the selection fits the limit. For a historical interval outside the recent tail, implement an explicit bounded time-index search before loading data.

This workflow does not repair or republish data. Original instrument files, derived operational Zarr stores, and validated scientific products are different data levels. The source-code MIT licence does not establish a licence to redistribute observations.